# Testes KPSS, ERS/DF-GLS e Zivot-Andrews

Este notebook cobre tres testes complementares ao ADF/PP para analise de raiz unitaria:

1. **KPSS (Kwiatkowski-Phillips-Schmidt-Shin)**: hipotese nula **invertida** - $H_0$: estacionariedade
2. **ERS/DF-GLS (Elliott-Rothenberg-Stock)**: versao com maior poder que o ADF, usando GLS detrending
3. **Zivot-Andrews**: raiz unitaria com quebra estrutural **endogena**

---

## Por que usar multiplos testes?

| Teste | $H_0$ | $H_1$ |
|-------|--------|--------|
| ADF/PP | Raiz unitaria | Estacionaria |
| **KPSS** | **Estacionaria** | **Raiz unitaria** |
| ERS | Raiz unitaria | Estacionaria |
| Zivot-Andrews | Raiz unitaria (com quebra) | Estacionaria |

A combinacao ADF + KPSS permite uma **estrategia de confirmacao**:
- ADF rejeita + KPSS nao rejeita $\Rightarrow$ **Estacionaria** (ambos concordam)
- ADF nao rejeita + KPSS rejeita $\Rightarrow$ **Raiz unitaria** (ambos concordam)
- Ambos rejeitam ou nao rejeitam $\Rightarrow$ **Inconclusivo** (possivelmente serie fracionariamente integrada)

---

## Conteudo

1. KPSS: teste de estacionariedade
2. ERS/DF-GLS: maior poder contra raiz unitaria
3. Zivot-Andrews: raiz unitaria com quebra estrutural
4. Comparacao conjunta ADF vs KPSS vs ERS vs ZA
5. Aplicacao ao PIB dos EUA e Brasil
6. Exercicios

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from chronobox.tests_stat import adf_test, kpss_test, ers_test, zivot_andrews_test

import sys
sys.path.insert(0, '..')
from utils.data_generators import (
    generate_unit_root_process,
    generate_structural_break,
    generate_trend_stationary,
)
from utils.plot_helpers import plot_unit_root_series, plot_structural_break

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Gerar series sinteticas para demonstracao
y_stationary = generate_unit_root_process(n=200, phi=0.7, seed=42, sigma=1.0)
y_unit_root = generate_unit_root_process(n=200, phi=1.0, seed=42, sigma=1.0)
df_break = generate_structural_break(n=200, break_point=0.5, shift=3.0, seed=42)
df_trend = generate_trend_stationary(n=200, trend_coef=0.05, seed=42)

# Visualizar
fig = plot_unit_root_series(
    {"I(0): Estacionaria": y_stationary,
     "I(1): Passeio Aleatorio": y_unit_root,
     "Com Quebra Estrutural": pd.Series(df_break['y'].values)},
    title="Series para Testes de Raiz Unitaria"
)
plt.show()

## 1. Teste KPSS

O teste KPSS decompoe a serie como:

$$y_t = \xi t + r_t + \varepsilon_t$$

onde $r_t = r_{t-1} + u_t$ e um passeio aleatorio. A hipotese nula e que $\text{Var}(u_t) = 0$, ou seja, o componente de passeio aleatorio e degenerado e a serie e **estacionaria**.

$$H_0: \text{a serie e estacionaria (nivel ou tendencia)}$$
$$H_1: \text{a serie possui raiz unitaria}$$

**ATENCAO**: A interpretacao e **invertida** em relacao ao ADF! Rejeitar $H_0$ no KPSS significa evidencia de raiz unitaria.

**Especificacoes:**
- `regression='c'`: testa estacionariedade em nivel ($H_0$: level-stationary)
- `regression='ct'`: testa estacionariedade ao redor de tendencia ($H_0$: trend-stationary)

In [ ]:
# KPSS na serie estacionaria I(0) - esperamos NAO rejeitar H0
print("=== KPSS: Serie Estacionaria I(0) ===")
result_kpss_i0 = kpss_test(y_stationary.values, regression='c')
print(result_kpss_i0.summary())
print(f"Decisao: {'Rejeita H0 (evidencia de raiz unitaria!)' if result_kpss_i0.reject_at_5pct else 'Nao rejeita H0 (serie e estacionaria)'}")

print("\n")

# KPSS na serie I(1) - esperamos REJEITAR H0
print("=== KPSS: Serie I(1) ===")
result_kpss_i1 = kpss_test(y_unit_root.values, regression='c')
print(result_kpss_i1.summary())
print(f"Decisao: {'Rejeita H0 (evidencia de raiz unitaria)' if result_kpss_i1.reject_at_5pct else 'Nao rejeita H0 (estacionaria)'}")

## 2. Teste ERS/DF-GLS

O teste ERS (Elliott, Rothenberg e Stock, 1996) e uma versao mais poderosa do ADF. Em vez de incluir termos deterministicos diretamente na regressao, ele primeiro **remove a tendencia via GLS (Generalized Least Squares)** e depois aplica o teste DF na serie de-trended.

Isso resulta em:
- **Maior poder** (menor probabilidade de erro tipo II)
- **Melhor desempenho** em amostras finitas
- Mesmas hipoteses do ADF: $H_0$: raiz unitaria, $H_1$: estacionaria

In [ ]:
# ERS/DF-GLS na serie estacionaria
print("=== ERS/DF-GLS: Serie Estacionaria I(0) ===")
result_ers_i0 = ers_test(y_stationary.values, regression='c', autolag='AIC')
print(result_ers_i0.summary())

print("\n")

# ERS na serie I(1)
print("=== ERS/DF-GLS: Serie I(1) ===")
result_ers_i1 = ers_test(y_unit_root.values, regression='c', autolag='AIC')
print(result_ers_i1.summary())

print("\n")

# ERS com tendencia na serie trend-stationary
print("=== ERS/DF-GLS: Trend-Stationary (ct) ===")
result_ers_ts = ers_test(df_trend['trend_stationary'].values, regression='ct', autolag='AIC')
print(result_ers_ts.summary())

## 3. Teste de Zivot-Andrews

O teste de Zivot-Andrews (1992) e um teste de raiz unitaria que permite uma **quebra estrutural endogena**. A data da quebra nao e especificada a priori - o teste encontra a data que minimiza a estatistica t (maximiza evidencia contra raiz unitaria).

$$H_0: \text{raiz unitaria sem quebra estrutural}$$
$$H_1: \text{estacionaria com uma quebra estrutural}$$

**Modelos disponiveis:**
- `model='a'`: quebra no intercepto (level shift)
- `model='b'`: quebra na tendencia (trend shift)
- `model='c'`: quebra no intercepto e na tendencia (ambos)

In [ ]:
# Zivot-Andrews na serie com quebra estrutural conhecida
y_break = df_break['y'].values
true_break = int(df_break['true_break_index'].iloc[0])

print(f"Quebra estrutural verdadeira no indice: {true_break}")
print(f"(ponto medio da serie de {len(y_break)} observacoes)\n")

# Testar com modelo de quebra no intercepto
print("=== Zivot-Andrews: Quebra no Intercepto (model='a') ===")
result_za_a = zivot_andrews_test(y_break, model='a', autolag='AIC')
print(result_za_a.summary())
if 'break_index' in result_za_a.additional_info:
    print(f"Quebra estimada: indice {result_za_a.additional_info['break_index']}")
    print(f"Quebra verdadeira: indice {true_break}")

print("\n")

# Testar com modelo completo (intercepto + tendencia)
print("=== Zivot-Andrews: Quebra no Intercepto e Tendencia (model='c') ===")
result_za_c = zivot_andrews_test(y_break, model='c', autolag='AIC')
print(result_za_c.summary())

In [ ]:
# Visualizar a serie com a quebra estimada pelo ZA
estimated_break = result_za_a.additional_info.get('break_index', true_break)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Serie com quebra verdadeira
plot_structural_break(pd.Series(y_break), true_break,
                      title=f'Quebra Verdadeira (indice={true_break})', ax=axes[0])

# Serie com quebra estimada pelo ZA
plot_structural_break(pd.Series(y_break), estimated_break,
                      title=f'Quebra Estimada pelo ZA (indice={estimated_break})', ax=axes[1])

plt.suptitle('Zivot-Andrews: Quebra Verdadeira vs Estimada', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Comparacao Conjunta: ADF vs KPSS vs ERS vs ZA

A estrategia recomendada e aplicar multiplos testes e interpretar conjuntamente. A tabela abaixo resume a logica:

In [ ]:
def bateria_testes(y, nome_serie):
    """Aplica bateria completa de testes e retorna DataFrame resumo."""
    results = []
    
    # ADF
    r = adf_test(y, regression='c', autolag='AIC')
    results.append({
        'Teste': 'ADF', 'H0': 'Raiz unitaria',
        'Estatistica': r.statistic, 'P-valor': r.pvalue,
        'Rejeita H0 (5%)': r.reject_at_5pct
    })
    
    # KPSS
    r = kpss_test(y, regression='c')
    results.append({
        'Teste': 'KPSS', 'H0': 'Estacionaria',
        'Estatistica': r.statistic, 'P-valor': r.pvalue,
        'Rejeita H0 (5%)': r.reject_at_5pct
    })
    
    # ERS
    r = ers_test(y, regression='c', autolag='AIC')
    results.append({
        'Teste': 'ERS/DF-GLS', 'H0': 'Raiz unitaria',
        'Estatistica': r.statistic, 'P-valor': r.pvalue,
        'Rejeita H0 (5%)': r.reject_at_5pct
    })
    
    # Zivot-Andrews
    r = zivot_andrews_test(y, model='c', autolag='AIC')
    results.append({
        'Teste': 'Zivot-Andrews', 'H0': 'RU sem quebra',
        'Estatistica': r.statistic, 'P-valor': r.pvalue,
        'Rejeita H0 (5%)': r.reject_at_5pct
    })
    
    df = pd.DataFrame(results)
    df['Estatistica'] = df['Estatistica'].map(lambda x: f"{x:.4f}")
    df['P-valor'] = df['P-valor'].map(lambda x: f"{x:.4f}" if x is not None else "N/A")
    
    print(f"\n{'=' * 70}")
    print(f"  Bateria de Testes: {nome_serie}")
    print(f"{'=' * 70}")
    print(df.to_string(index=False))
    
    # Interpretacao conjunta
    adf_rejects = results[0]['Rejeita H0 (5%)']
    kpss_rejects = results[1]['Rejeita H0 (5%)']
    
    if adf_rejects and not kpss_rejects:
        print("\n>> Conclusao: ESTACIONARIA (ADF rejeita RU, KPSS nao rejeita estacionariedade)")
    elif not adf_rejects and kpss_rejects:
        print("\n>> Conclusao: RAIZ UNITARIA (ADF nao rejeita RU, KPSS rejeita estacionariedade)")
    elif adf_rejects and kpss_rejects:
        print("\n>> Conclusao: INCONCLUSIVO (ambos rejeitam - possivelmente fracionaria)")
    else:
        print("\n>> Conclusao: INCONCLUSIVO (nenhum rejeita)")
    
    return df

In [ ]:
# Aplicar bateria em series sinteticas
_ = bateria_testes(y_stationary.values, "Serie Estacionaria I(0)")
_ = bateria_testes(y_unit_root.values, "Serie com Raiz Unitaria I(1)")
_ = bateria_testes(y_break, "Serie com Quebra Estrutural")

## 5. Aplicacao: PIB dos EUA e Brasil

Vamos aplicar a bateria completa de testes aos dados reais.

In [ ]:
# Carregar dados
gdp_us = pd.read_csv('../data/us_gdp_quarterly.csv', parse_dates=['date'], index_col='date')
gdp_br = pd.read_csv('../data/brazil_gdp.csv', parse_dates=['date'], index_col='date')

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

axes[0, 0].plot(gdp_us.index, gdp_us['log_gdp'], color='steelblue')
axes[0, 0].set_title('Log PIB EUA (nivel)')
axes[0, 0].grid(True, alpha=0.3)

axes[0, 1].plot(gdp_us['gdp_growth'].dropna(), color='steelblue')
axes[0, 1].set_title('Crescimento PIB EUA (%)')
axes[0, 1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)
axes[0, 1].grid(True, alpha=0.3)

axes[1, 0].plot(gdp_br.index, gdp_br['log_gdp'], color='coral')
axes[1, 0].set_title('Log PIB Brasil (nivel)')
axes[1, 0].grid(True, alpha=0.3)

axes[1, 1].plot(gdp_br['gdp_growth'].dropna(), color='coral')
axes[1, 1].set_title('Crescimento PIB Brasil (%)')
axes[1, 1].axhline(y=0, color='black', linestyle='--', linewidth=0.5)
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Bateria de testes - PIB EUA
_ = bateria_testes(gdp_us['log_gdp'].values, "Log PIB EUA (nivel)")
_ = bateria_testes(np.diff(gdp_us['log_gdp'].values), "Log PIB EUA (1a diferenca)")

In [ ]:
# Bateria de testes - PIB Brasil (com quebra estrutural ~2003)
_ = bateria_testes(gdp_br['log_gdp'].values, "Log PIB Brasil (nivel)")
_ = bateria_testes(np.diff(gdp_br['log_gdp'].values), "Log PIB Brasil (1a diferenca)")

# Zivot-Andrews especifico para Brasil (esperamos que detecte a quebra)
print("\n" + "=" * 70)
print("  Zivot-Andrews detalhado: PIB Brasil")
print("=" * 70)
za_br = zivot_andrews_test(gdp_br['log_gdp'].values, model='a', autolag='AIC')
print(za_br.summary())
if 'break_index' in za_br.additional_info:
    break_idx = za_br.additional_info['break_index']
    print(f"Quebra detectada no indice {break_idx} ({gdp_br.index[break_idx].strftime('%Y-%m')})")

## 6. Tabela Resumo Consolidada

Vamos reunir os resultados de todos os testes em uma unica tabela para comparacao.

In [ ]:
# Tabela resumo consolidada de todos os testes
all_results = []

series_dict = {
    "I(0) sintetica": y_stationary.values,
    "I(1) sintetica": y_unit_root.values,
    "Quebra estrutural": y_break,
    "Log PIB EUA": gdp_us['log_gdp'].values,
    "Diff Log PIB EUA": np.diff(gdp_us['log_gdp'].values),
    "Log PIB Brasil": gdp_br['log_gdp'].values,
    "Diff Log PIB Brasil": np.diff(gdp_br['log_gdp'].values),
}

for nome, y in series_dict.items():
    r_adf = adf_test(y, regression='c', autolag='AIC')
    r_kpss = kpss_test(y, regression='c')
    r_ers = ers_test(y, regression='c', autolag='AIC')
    r_za = zivot_andrews_test(y, model='c', autolag='AIC')
    
    all_results.append({
        'Serie': nome,
        'ADF stat': f"{r_adf.statistic:.3f}",
        'ADF p': f"{r_adf.pvalue:.3f}",
        'KPSS stat': f"{r_kpss.statistic:.3f}",
        'KPSS p': f"{r_kpss.pvalue:.3f}" if r_kpss.pvalue is not None else "N/A",
        'ERS stat': f"{r_ers.statistic:.3f}",
        'ZA stat': f"{r_za.statistic:.3f}",
        'Conclusao': (
            'I(0)' if r_adf.reject_at_5pct and not r_kpss.reject_at_5pct
            else 'I(1)' if not r_adf.reject_at_5pct and r_kpss.reject_at_5pct
            else 'Inconcl.'
        )
    })

df_all = pd.DataFrame(all_results)
print("=" * 100)
print("  TABELA RESUMO: Bateria Completa de Testes de Raiz Unitaria")
print("=" * 100)
print(df_all.to_string(index=False))

## Exercicio

### Exercicio 1: Estrategia de Confirmacao ADF-KPSS

Gere uma serie com `phi=0.95` (quase raiz unitaria, mas estacionaria) usando `generate_unit_root_process`:
1. Aplique ADF e KPSS. Os testes concordam?
2. Aplique ERS (que tem maior poder). Ele detecta a estacionariedade?
3. Repita com amostras de tamanho n=50, 100, 200, 500. Como o tamanho da amostra afeta os resultados?

**Dica**: `generate_unit_root_process(n=200, phi=0.95, seed=42)`

In [ ]:
# Exercicio 1 - Seu codigo aqui
# Gere series com phi=0.95 e aplique a bateria de testes


### Exercicio 2: Zivot-Andrews no PIB do Brasil

O PIB do Brasil possui uma quebra estrutural por volta de 2003 (mudanca de regime economico):
1. Aplique o Zivot-Andrews com os tres modelos ('a', 'b', 'c')
2. Compare as datas de quebra estimadas
3. Qual modelo e mais adequado para o caso brasileiro?
4. O resultado do ZA muda a conclusao obtida pelo ADF convencional?

**Dica**: O ADF pode falhar em rejeitar raiz unitaria quando ha quebra estrutural (vies de nao-rejeicao)

In [ ]:
# Exercicio 2 - Seu codigo aqui
# Aplique Zivot-Andrews com modelos 'a', 'b', 'c' no PIB Brasil
